#rev.6




rev.3
ハイパラメータ調整
オリジナルコード長期化のため

===== /tmp/agentbench-results/dbbench/overall.json ===== { "total": 150, "validation": { "running": 0.0, "completed": 0.4666666666666667, "agent context limit": 0.0, "agent validation failed": 0.4866666666666667, "agent invalid action": 0.0, "task limit reached": 0.04666666666666667, "unknown": 0.0, "task error": 0.0, "average_history_length": 8.16, "max_history_length": 24, "min_history_length": 4 }, "custom": { "other_accuracy": 0.0, "counting_accuracy": 0.0, "comparison_accuracy": 0.1111111111111111, "ranking_accuracy": 0.3, "aggregation-SUM_accuracy": 0.0, "aggregation-MIN_accuracy": 0.2, "aggregation-MAX_accuracy": 0.3333333333333333, "aggregation-AVG_accuracy": 0.0, "SELECT_accuracy": 0.11475409836065574, "INSERT_accuracy": 0.3333333333333333, "UPDATE_accuracy": 0.7, "overall_cat_accuracy": 0.382695810564663 } } ===== /tmp/agentbench-results/alfworld/overall.json ===== { "total": 50, "validation": { "running": 0.0, "completed": 0.28, "agent context limit": 0.0, "agent validation failed": 0.0, "agent invalid action": 0.5, "task limit reached": 0.22, "unknown": 0.0, "task error": 0.0, "average_history_length": 62.48, "max_history_length": 91, "min_history_length": 21 }, "custom": { "overall": { "total": 50, "pass": 14, "wrong": 36, "success_rate": 0.28 } } } 2.5488


#rev.3 -> rev.4
（Gemini 2026/2/25 9：45）

1. 提案A：SFT_OUTPUT_MARKERS の最小化 【最優先・大賛成】
評価：★★★★★（いますぐ適用すべき）
DBBenchの validation failed の直接的な原因はこれである可能性が高いです。現状の設定（Final:,Answer:,Result:,Response:）だと、モデルは回答の前にこれらのマーカーを付ける癖を学習してしまいます。評価スクリプトは純粋なSQLを期待しているため、「Final: SELECT * FROM...」と出力した瞬間に不正解（パース失敗）となります。
結論: アドバイス通り os.environ["SFT_OUTPUT_MARKERS"] = "Output:" のみに絞る（あるいは空にする）のは非常に有効です。

2. 提案D：SFT_MASK_COT = "1" にする 【ALFWorld向けに強く推奨】
評価：★★★★★（いますぐ適用すべき）
ALFWorldの invalid action (50%) に対する特効薬になる可能性があります。学習データにCoT（思考プロセス）が含まれている場合、モデルが「推論環境でも思考プロセスをそのままアクションとして出力してしまう」現象が起きています。MASK_COT = "1" にすることで、思考部分の損失（Loss）を計算せず、最終的な「アクションの出力」にフォーカスして学習させることができるため、余計なおしゃべりが減ります。

3. 提案C：SFT_LR（学習率）を下げる 【推奨】
評価：★★★★☆（提案A・Dとセットで試すべき）
フォーマットが崩れるのは「事前学習モデルが元々持っていた命令追従能力が、強すぎるSFTによって上書き破壊された（破局的忘却）」サインでもあります。現在の 1e-5 から 5e-6 に下げるのは、過学習を防ぎ、ベースモデルの賢さを残すための安全で定石的なアプローチです。


#rev.4
===== /tmp/agentbench-results/dbbench/overall.json =====
{
  "total": 150,
  "validation": {
    "running": 0.0,
    "completed": 0.8533333333333334,
    "agent context limit": 0.0,
    "agent validation failed": 0.12,
    "agent invalid action": 0.0,
    "task limit reached": 0.02666666666666667,
    "unknown": 0.0,
    "task error": 0.0,
    "average_history_length": 7.533333333333333,
    "max_history_length": 24,
    "min_history_length": 4
  },
  "custom": {
    "other_accuracy": 0.2857142857142857,
    "counting_accuracy": 0.36363636363636365,
    "comparison_accuracy": 0.3333333333333333,
    "ranking_accuracy": 0.5,
    "aggregation-SUM_accuracy": 0.16666666666666666,
    "aggregation-MIN_accuracy": 0.2,
    "aggregation-MAX_accuracy": 0.16666666666666666,
    "aggregation-AVG_accuracy": 0.42857142857142855,
    "SELECT_accuracy": 0.32786885245901637,
    "INSERT_accuracy": 0.36231884057971014,
    "UPDATE_accuracy": 0.7,
    "overall_cat_accuracy": 0.4633958976795755
  }
}

===== /tmp/agentbench-results/alfworld/overall.json =====
{
  "total": 50,
  "validation": {
    "running": 0.0,
    "completed": 0.32,
    "agent context limit": 0.0,
    "agent validation failed": 0.0,
    "agent invalid action": 0.28,
    "task limit reached": 0.4,
    "unknown": 0.0,
    "task error": 0.0,
    "average_history_length": 66.8,
    "max_history_length": 91,
    "min_history_length": 21
  },
  "custom": {
    "overall": {
      "total": 50,
      "pass": 16,
      "wrong": 34,
      "success_rate": 0.32
    }
  }
}

3.0131


#rev.4 -> rev.5
（Gemini 2026/2/25 13:45）

Gemini の回答
いただいたログファイル（runs.json）とベンチマークの評価結果を詳細に解析しました。

まず、DBBenchが劇的に改善（completed: 0.46 → 0.85、validation failed: 0.48 → 0.12） したのは素晴らしい進捗です。これは前回アドバイスした「提案A（マーカーの最小化）」と「提案C（学習率を少し下げる）」が完璧に機能し、不要な文字列（前置きなど）を出力するフォーマット崩壊を食い止めた結果です。

一方で、Alfworldのスコアが伸び悩んでいる原因もログから明確に判明しました。

📊 ログ解析から判明したAlfworldの失敗原因
invalid action（存在しない行動による即死）は半減しましたが、代わりに task limit reached（制限ターン到達による時間切れ）が 0.22 → 0.40 に倍増しています。ログを確認したところ、以下のような致命的なループが発生していました。

状況を把握できず同じ行動を繰り返す
「drawer 1 を開ける → 調べる → 閉じる → また開ける」といった無意味な行動を数十ターンにわたって繰り返しています。

勝手に「諦める」宣言をしてループする
次に何をすればいいか分からなくなったモデルが、用意されているコマンド一覧（admissible_commands）を無視して、勝手に ACTION: Task failed. や ACTION: Task succeeded. という文字列を出力して終了しようとするケースが頻発しています（環境側はこれを不正なコマンドとして弾くため、結果的に制限時間切れまでループします）。

💡 なぜこれが起きたのか？（原因と対策）
これは、前回導入した SFT_MASK_COT = "1" の副作用と、コンテキスト長の不足が組み合わさって起きています。

DBBenchのような「一発でSQLを出すタスク」には CoT（思考プロセス）のマスクが有効でしたが、Alfworldのような「現状を整理し、過去の行動を記憶しながら進める多段階の探索タスク」では、思考（THOUGHT:）部分を学習対象から外してしまうと、「自分が今どこにいて、何を持っているのか」という状況認識力が育たず、支離滅裂な行動や諦めに繋がります。

また、Alfworldはやり取りの履歴が非常に長いため、SFT_MAX_SEQ_LEN が 1024 では過去の記憶が途中で切り捨てられてしまい、「さっき調べた場所を忘れてまた調べる」というループを引き起こします。

🚀 目標5.0を達成するための次の一手
DBBenchの成功状態をキープしつつ、Alfworldの推論能力を取り戻すために、設定を以下のハイブリッド型に調整してください。

1. MASK_COT を無効化し、思考プロセスを学習させる（最優先）
AlfworldのReAct（推論と行動）能力を取り戻すため、思考部分も学習対象に戻します。

Python
os.environ["SFT_MASK_COT"] = "0"
2. コンテキスト長（MAX_SEQ_LEN）を拡張し、記憶喪失を防ぐ
コメントアウトされていた元の設定を活かし、長文の履歴を保持できるように 2048 に引き上げます。これにより、堂々巡りのループを劇的に減らすことができます。

Python
os.environ["SFT_MAX_SEQ_LEN"] = "2048"
3. 成功したDBBench向け設定は「維持」する
今回の改善の立役者であるこの2つの設定は、絶対に現状（変更後）のまま維持してください。

Python
os.environ["SFT_OUTPUT_MARKERS"] = "Output:"
os.environ["SFT_LR"] = "5e-6"
（オプション）
もしこれでもAlfworldが賢くなりきらない場合は、モデルが複雑なタスクの解き方を学習しきれていない（学習不足）可能性があるため、os.environ["SFT_EPOCHS"] = "3" に1つだけ上げることも検討の余地がありますが、まずは上記「1」と「2」の変更のみで回していただくのが最も安全で確実なアプローチです。




{
    "total": 150,
    "validation": {
        "running": 0.0,
        "completed": 0.8733333333333333,
        "agent context limit": 0.0,
        "agent validation failed": 0.10666666666666667,
        "agent invalid action": 0.0,
        "task limit reached": 0.02,
        "unknown": 0.0,
        "task error": 0.0,
        "average_history_length": 8.013333333333334,
        "max_history_length": 34,
        "min_history_length": 4
    },
    "custom": {
        "other_accuracy": 0.2857142857142857,
        "counting_accuracy": 0.2727272727272727,
        "comparison_accuracy": 0.5555555555555556,
        "ranking_accuracy": 0.4,
        "aggregation-SUM_accuracy": 0.16666666666666666,
        "aggregation-MIN_accuracy": 0.2,
        "aggregation-MAX_accuracy": 0.16666666666666666,
        "aggregation-AVG_accuracy": 0.7142857142857143,
        "SELECT_accuracy": 0.36065573770491804,
        "INSERT_accuracy": 0.34782608695652173,
        "UPDATE_accuracy": 0.7,
        "overall_cat_accuracy": 0.46949394155381324
    }
}
{
    "total": 50,
    "validation": {
        "running": 0.0,
        "completed": 0.32,
        "agent context limit": 0.0,
        "agent validation failed": 0.0,
        "agent invalid action": 0.46,
        "task limit reached": 0.22,
        "unknown": 0.0,
        "task error": 0.0,
        "average_history_length": 63.44,
        "max_history_length": 91,
        "min_history_length": 21
    },
    "custom": {
        "overall": {
            "total": 50,
            "pass": 16,
            "wrong": 34,
            "success_rate": 0.32
        }
    }
}


#rev.5 ->rev.6
(2026/2/25 16:20)

分析の結果、DBBenchは「正解の抽出（Action: Answer）」という点ではほぼ完成（87.3%）していますが、ALFWorldが「存在しないコマンドを試して自滅する（invalid action）」ことでスコアを大きく下げていることがわかりました。

特に最新の実行結果では、MASK_COT = 0（思考プロセスを学習する）にしたことで、モデルが「DBBench（SQLタスク）の考え方」に染まってしまい、ALFWorldにおいて**「目の前の選択肢（AVAILABLE ACTIONS）を無視して、自分の思い込みで行動（inventoryやhelpなど）を生成する」**という悪影響（過学習・干渉）が出ています。

目標の「5.0突破（または平均0.5）」を達成するために、DBBenchの精度を維持しつつALFWorldを救い出すための設定を提案します。

🚀 パフォーマンス向上のための修正提案
1. SFT_MASK_COT = "1" に戻す 【最重要】
前回の結果から、思考プロセス（THOUGHT）を学習させるとALFWorldでの invalid action が 28% → 46% に急増しました。これは、DBBench専用のデータセットに含まれる「考え方の癖」が、ALFWorldの汎用的な推論能力を邪魔しているためです。
思考部分をマスクすることで、DBBenchの「最終的なSQL形式」だけを効率よく学ばせ、ALFWorldではモデルが元々持っている「指示に従う能力」を保護します。

2. SFT_LORA_R = "16" / SFT_LORA_ALPHA = "32" に下げる
現状の R=32 は、4Bという小規模なモデルに対しては「学習の自由度」が高すぎ、DBBenchの形式に過剰適合（オーバーフィット）して他のタスクを壊してしまう原因になります。ランクを下げることで、モデルの汎用性を維持（レギュラリゼーション）します。

3. SFT_LR = "3e-6" に微調整
DBBenchはすでに高い完成度にあるため、学習率をさらに少し下げることで、ALFWorldの「元々の賢さ」を壊さないように慎重に微調整します。

4. SFT_MAX_SEQ_LEN = "2048" は維持
ログを確認したところ、ALFWorldの後半で「同じ場所を何度も調べる」ループがまだ見られます。コンテキスト長は長い方が有利なため、これは維持します。

In [1]:
# -----------------------------
# Module2:
# -----------------------------
import os
import shutil
from datetime import datetime
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
from datetime import datetime
from google.colab import drive, userdata

# 再定義: ノートブックのパスなどの設定（念のため再読み込み）
GIT_USERNAME  = "kumeichit"
GIT_EMAIL     = "htakeda@km-soft.net"
# REPO_URLを正しいGitHubリポジトリの形式に修正。スペースをハイフンに、末尾の#を除去。
REPO_URL      = "https://github.com/kumeichit/LLM_Colab_Qwen3-4B-Instruct-2507_Kadai-Code-DBBench_ALFWorld.git"

NOTEBOOK_PATH = "/content/drive/MyDrive/Colab Notebooks/LLD_FINAL_ADVANCE_ORI/2025最終課題アドバンスドコンペ標準コード_SFT.ipynb"

def save_and_push(commit_msg: str):
    """
    現在のノートブックとSFTパラメータ設定をGitHubにPushします。
    改行を含むメッセージ(トリプルクォート)にも対応しています。
    """
    print(f"[GitHub] Processing commit...")

    # 1. 認証トークンの取得
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        # Secret未設定時は入力させる
        token = input("Enter your GitHub Token: ").strip()

    if not token:
        print("[Error] GitHub Token is required.")
        return

    # 2. リポジトリの準備
    # REPO_URLの修正に伴い、.replace(".git", "")は不要になったため削除
    repo_name = REPO_URL.split("/")[-1]
    repo_dir = f"/content/git_work/{repo_name}"
    auth_url = REPO_URL.replace("https://", f"https://{token}@")

    if not os.path.exists(repo_dir):
        print(f"[GitHub] Cloning repository...")
        !git clone {auth_url} {repo_dir}
        !git config --global user.email "{GIT_EMAIL}"
        !git config --global user.name "{GIT_USERNAME}"
    else:
        print(f"[GitHub] Pulling latest changes...")
        %cd {repo_dir}
        !git pull origin main || git pull origin master

    # 3. ノートブックのコピー
    if os.path.exists(NOTEBOOK_PATH):
        dest_nb = os.path.join(repo_dir, os.path.basename(NOTEBOOK_PATH))
        !cp "{NOTEBOOK_PATH}" "{dest_nb}"
    else:
        print(f"[Warning] Notebook not found at: {NOTEBOOK_PATH}\nMake sure the path is correct and Drive is mounted.")

    # 4. パラメータ(環境変数)の保存
    config_data = {k: v for k, v in os.environ.items() if k.startswith("SFT_")}
    config_path = os.path.join(repo_dir, "sft_config_dump.json")
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config_data, f, indent=2, ensure_ascii=False)

    # 5. Commit & Push
    %cd {repo_dir}
    !git add .

    status = !git status --porcelain
    if not status:
        print("[GitHub] No changes to commit.")
        return

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    full_msg = f"{commit_msg}\n\n(Timestamp: {timestamp})"

    # メッセージをファイルに保存して、改行や特殊文字によるシェルエラーを回避
    msg_file = "COMMIT_MSG_TEMP.txt"
    with open(msg_file, "w", encoding="utf-8") as f:
        f.write(full_msg)

    try:
        !git commit -F {msg_file}
        print(f"[GitHub] Pushing...")
        !git push origin main || !git push origin master
        print(f"[GitHub] ✅ Done! Pushed.")
    finally:
        if os.path.exists(msg_file):
            os.remove(msg_file)

# === 再実行 ===
save_and_push (
    """
    rev.2 3.4...

""")

[GitHub] Processing commit...
[GitHub] Cloning repository...
Cloning into '/content/git_work/LLM_Colab_Qwen3-4B-Instruct-2507_Kadai-Code-DBBench_ALFWorld.git'...
/content/git_work/LLM_Colab_Qwen3-4B-Instruct-2507_Kadai-Code-DBBench_ALFWorld.git
[main (root-commit) bf7ad23]     rev.2 3.4...
 2 files changed, 2 insertions(+)
 create mode 100644 "2025\346\234\200\347\265\202\350\252\262\351\241\214\343\202\242\343\203\210\343\202\231\343\203\217\343\202\231\343\203\263\343\202\271\343\203\210\343\202\231\343\202\263\343\203\263\343\203\230\343\202\232\346\250\231\346\272\226\343\202\263\343\203\274\343\203\210\343\202\231_SFT.ipynb"
 create mode 100644 sft_config_dump.json
[GitHub] Pushing...
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 12 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 46.95 KiB | 6.71 MiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/kumeichit/LLM_Colab_Q

# エージェント Trajectory SFT（Unsloth / Colab L4）標準学習コード：実行ガイド

本ノートブックは、**AgentBench系タスク（ALFWorld・DBBench）のエージェント性能向上**を目的として、  
小型LLM（Qwen3-4B Instruct-2507）に対して **SFT（Supervised Fine-Tuning）** を行う標準コードです。

学習は **Unsloth + LoRA** を利用し、**Colab L4 GPU**で動作するように最適化されています。


## 1. このコードが行うこと（概要）



このコードは大きく3段階で構成されています。

1. **環境固定（依存パッケージのバージョン固定）**  
   Colabの環境変化による不具合を避けるため、numpy/transformers/trl/unsloth等を特定バージョンで揃えます。

2. **SFT（教師あり微調整）の実行**  
   Hugging Face Hub 上のトラジェクトリデータセット（ALFWorld / DBBench）を読み込み、ベースモデルに LoRA アダプタを差し込み、学習します。  
   学習の損失（loss）は **全assistant turn**にかかる設計です（マルチターンのtool-useパターンを学習させやすい）。

3. **（任意）マージ済みモデルのHugging Faceへのアップロード**  
   学習で得られた LoRA アダプタをベースモデルにマージし、HF Hub に保存できます。

---



## 2. 実行手順（最短手順）



### Step 0: Colab の準備
- ランタイムの種類を **GPU** に変更し、GPU が **L4** になっていることを確認してください。
- 過去の実行で環境が壊れている場合は **Runtime > Factory reset** を推奨します。

### Step 1: 依存関係インストール
- `uv pip install` を上から順に実行します。
- 実行後、バージョン表示が想定通りであることを確認します（`unsloth import OK` が出ること）。

### Step 2: Hugging Face へログイン
- Colabの秘密鍵サービス（🔑アイコン）にHFトークンを設定しておくと、自動でログインされます。
- 秘密鍵が未設定の場合は警告が表示されます。

### Step 3: 学習の実行
- `main()` が呼ばれ、学習が開始します。
- 学習中に `[LabelStats:train]` が表示されます。これは「loss対象トークンが極端にゼロになっていないか」の健康診断です。
- 学習前に `filter_has_supervision` が実行され、教師信号のないサンプルが自動除外されます。

### Step 4: 学習成果物の確認
- 学習後、`OUT_LORA_DIR` に以下が保存されます（最低限）：
  - `adapter_config.json`
  - `adapter_model.safetensors`（または `adapter_model.bin`）
  - tokenizer 関連ファイル

### Step 5:（任意）モデルマージとアップロード
- LoRAアダプタをベースモデルにマージして、完全な推論用モデルを生成できます。
- マージ済みモデルをHugging Faceにアップロードできます。

---

## 3. 出力（何が生成されるか）



- `OUT_LORA_DIR`（例：`/content/lora_structeval_t_qwen3_4b`）に、
  **LoRAアダプタ（差分重み）**が保存されます。
- このアダプタをベースモデルに適用（またはマージ）して推論することで、ALFWorld・DBBenchタスクのスコア改善を狙います。

---



## 4. 学習データセットの説明



### 4.1 データセット概要
本コードでは、以下の2種類のトラジェクトリデータセットを使用できます。
環境変数 `SFT_DATASET_ID` で切り替えます。

#### A. ALFWorld トラジェクトリデータセット
- HF Dataset: `u-10bei/sft_alfworld_trajectory_dataset_v5`

ALFWorld（テキストベースの家庭内タスク環境）におけるエージェントのトラジェクトリです。
エージェントが「フライパンをダイニングテーブルに置く」「ライトで時計を調べる」などの
家庭内タスクを、テキストコマンドの発行と環境からの観察の繰り返しで解決します。

- 行数：**約 2,502 件**（成功＋失敗トラジェクトリ）
- タスクタイプ：Pick & Place / Clean & Place / Heat & Place / Cool & Place / Examine in Light / Pick Two & Place

#### B. DBBench トラジェクトリデータセット
- HF Dataset: `u-10bei/sft_sql_dataset_trajectories_v5`

DBBench（AgentBenchのデータベース操作タスク）におけるエージェントのトラジェクトリです。
ユーザーの質問に対して、スキーマ探索→SQL構築→実行→エラー回復→最終回答という
ReActスタイルの対話フローを辿ります。

- 行数：**約 3,000 件**
- DBスキーマ：6種（university_academic / online_retail / hospital_management / company_hr / music_streaming / city_library）
- タスクタイプ：Querying / Analysis / Updating（3難易度レベル）
- 対話形式：ReAct形式（50%）＋ function-calling形式（50%）
- スキーマ探索付き：約60%、エラー回復付き：約15%

### 4.2 共通のデータ形式
両データセットとも **マルチターンのmessages形式** です。
`system`（システムプロンプト）→ `user`（質問・環境観察）→ `assistant`（思考＋アクション）→ `tool`（実行結果）の繰り返しで構成されます。

#### DBBench（ReAct形式）の例：
```json
[
  {"role": "system", "content": "You are a database agent that interacts with a MySQL database..."},
  {"role": "user", "content": "Find all classrooms in 'Science Hall' with capacity > 100."},
  {"role": "assistant", "content": "Thought: I should first explore the schema...\nAction: execute_sql\n```sql\nSHOW TABLES;\n```"},
  {"role": "tool", "content": "Classrooms | Courses | Departments | ..."},
  {"role": "assistant", "content": "Thought: Now let me query the Classrooms table...\nAction: execute_sql\n```sql\nSELECT room_number, capacity FROM Classrooms WHERE building_name = 'Science Hall' AND capacity > 100;\n```"},
  {"role": "tool", "content": "room_number | capacity\n301 | 150\n..."},
  {"role": "assistant", "content": "Thought: I have the results.\nAnswer: The classrooms in Science Hall with capacity > 100 are: Room 301 (150 seats)..."}
]
```

#### ALFWorld の例：
```json
[
  {"role": "user", "content": "You are in the middle of a room...Your task is to: put a hot apple in fridge."},
  {"role": "assistant", "content": "Think: I need to find an apple first.\nAction: go to countertop 1"},
  {"role": "tool", "content": "On the countertop 1, you see an apple 1, a knife 2."},
  ...
]
```

### 4.3 メインコンペの構造化出力データセットとの主な違い
- **ターン数**：2ターン → 数ターン〜数十ターン（マルチターンtool-use）
- **role**：user/assistant のみ → system/user/assistant/tool の4種
- **学習対象**：最終応答のみ → 全assistant turn（中間ステップ含む）
- **シーケンス長**：512 → 2048（長いトラジェクトリに対応）

# 実行コード

## Step 1:依存関係インストール

In [2]:
# ============================================================
# 0) 依存関係の固定（Colabの“環境ブレ”対策）
# ============================================================
# Colab（無料版）は、ある日突然プリインストール版が変わり、
# それまで動いていた学習コードが壊れることが頻繁にあります。
# そのため、このセルでは「一度全部消す → 互換が確認できたバージョンを入れ直す」
# という“強制的な再現性確保”をしています。
#
# やりがちな事故：
# - 既に入っているパッケージと混ざって「importは通るが実行で落ちる」
# - transformers / trl / unsloth の相性不一致で、謎エラーや速度低下が起きる
#
# ※このセルは“おまじない”ではなく「再現性のための重要工程」です。

!uv pip install "numpy==2.0.2" "pandas==2.2.2"

# unsloth-zoo が要求する範囲に合わせる
# ここでは Unsloth と相性の良い transformers / trl / accelerate / peft / bitsandbytes を固定します。
# 特に transformers は細かいバージョン差で挙動が変わりやすいので固定が重要です。
!uv pip install \
  "datasets==4.3.0" \
  "trl==0.24.0" \
  "transformers==4.56.2" \
  "accelerate==1.4.0" \
  "peft==0.13.2" \
  "bitsandbytes==0.45.0"

# unsloth / zoo を同系列で揃える（zoo側の要求に合わせる）
# Unsloth本体と unsloth-zoo は“セット運用”が基本です。片方だけ上げると壊れがちです。
!uv pip install "unsloth-zoo==2025.12.7" "unsloth==2025.12.7"



Using Python 3.12.12 environment at: /usr
Audited 2 packages in 160ms
Using Python 3.12.12 environment at: /usr
Resolved 70 packages in 434ms
Prepared 8 packages in 1.54s
Uninstalled 6 packages in 372ms
Installed 8 packages in 49ms
 - accelerate==1.12.0
 + accelerate==1.4.0
 + bitsandbytes==0.45.0
 - datasets==4.0.0
 + datasets==4.3.0
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.2
 - peft==0.18.1
 + peft==0.13.2
 - pyarrow==18.1.0
 + pyarrow==23.0.1
 - transformers==5.0.0
 + transformers==4.56.2
 + trl==0.24.0
Using Python 3.12.12 environment at: /usr
Resolved 88 packages in 156ms
Prepared 9 packages in 749ms
Uninstalled 2 packages in 22ms
Installed 9 packages in 13ms
 - bitsandbytes==0.45.0
 + bitsandbytes==0.49.2
 + cut-cross-entropy==25.1.1
 + hf-transfer==0.1.9
 + msgspec==0.20.0
 - torchao==0.10.0
 + torchao==0.16.0
 + tyro==1.0.8
 + unsloth==2025.12.7
 + unsloth-zoo==2025.12.7
 + xformers==0.0.35


In [3]:
# ============================================================
# 0.1) バージョン確認（“動くはず”の状態かを目視で確認）
# ============================================================
# ここで想定バージョンとズレている場合、
# 後工程で原因不明のエラーが出る確率が一気に上がります。

from unsloth import FastLanguageModel
import numpy as np, pandas as pd
import datasets, trl, transformers, torch

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)

print("unsloth import OK")

# 期待値：
# numpy 2.0.2
# pandas 2.2.2
# datasets 4.3.0（または <4.4.0 で 4.0.* / 4.1.0 以外）
# trl 0.24.0（または 0.18.2〜0.24.0 で 0.19.0以外）
# unsloth import OK


# -----------------------------
# 0) Install (single cell)
# -----------------------------
# NOTE:
# - Colabは初期状態が頻繁に変わるため、ピン留めで安定化します。
#   もし依存関係が壊れている環境であれば、Runtime > Factory reset を推奨。

# このセルを実行して、上の「期待値」にもしなっていない場合は、下記のコメントアウトを外して実行してみてください。
# !uv pip install \
#   "numpy==2.0.2" \
#   "pandas==2.2.2" \
#   "datasets==4.3.0" \
#   "trl==0.24.0" \
#   "transformers==4.57.3" \又は、4.56.2
#   "accelerate==1.4.0" \
#   "peft==0.13.2" \
#   "bitsandbytes==0.45.0" \
#   "unsloth-zoo==2025.12.7" \
#   "unsloth==2025.12.7" \
#   "huggingface_hub"


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1825: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


🦥 Unsloth Zoo will now patch everything to make training faster!
numpy 2.0.2
pandas 2.2.2
datasets 4.3.0
trl 0.24.0
transformers 4.56.2
torch 2.10.0+cu128
unsloth import OK


## Step 2: HuggingFace ログイン

Colabの秘密鍵サービス（左パネルの🔑アイコン）に `HF_TOKEN` を設定しておくと、自動的にログインされます。
秘密鍵サービスの設定手順：
1. Hugging Face にログイン（https://huggingface.co/）
2. Settings → Access Tokens からトークンを作成（Write権限推奨）
3. Colabの左パネル 🔑 → 新しい秘密鍵を追加 → 名前: `HF_TOKEN`、値: コピーしたトークン

In [4]:
import os
from google.colab import userdata

# HF_TOKENを環境変数またはColabの秘密鍵から取得
# Colabの左側のパネルにある「秘密鍵（🔑）」アイコンをクリックし、
# 「HF_TOKEN」という名前でトークンを設定してください。
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = userdata.get('HF_TOKEN') # 秘密鍵サービスに登録した名前に変更

if not HF_TOKEN:
    print("警告: HF_TOKENが見つかりません。Hugging Faceへのログインが失敗する可能性があります。")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=True)

In [5]:

# -----------------------------
# 1) ライブラリ再読み込み
# -----------------------------
# Cell 12 で既にimport済みですが、Colabでセルを個別に実行する場合に備えて
# 必要なライブラリを再度importしています。

from unsloth import FastLanguageModel
import numpy as np, pandas as pd
import datasets, trl, transformers, torch

## Step3:学習の実行

============================================================  
2) Training code  
============================================================  
ここからがSFT本体です。
大まかな流れ：
1) 設定値（モデル名、データセット、LoRA設定、学習率など）を読み込む
2) データセットをHFから取得し、必要な形（messages形式）を満たすものだけ残す
3) tokenizerで「学習に使うテキスト」を作ってキャッシュする（高速化）
4) ベースモデルを4bitでロードし、LoRAアダプタを差し込む
5) Trainerで学習を回す
6) LoRAアダプタを保存する

- ベースモデル：Qwen3-4B-Instruct-2507
- GPU：L4（24GB VRAM、bf16ネイティブ対応）を前提としています。
- 学習方式：LoRA（ベースモデルをfull精度でロードし、LoRAアダプタのみ学習）
  - `load_in_4bit=False` でベースモデルを読み込み、LoRAアダプタ（軽量差分）だけを学習します。
  - L4はbf16をネイティブサポートしているため、`bf16=True` で高速かつ安定した学習が可能です。
  - そのため、学習後に保存されるのも「アダプタ」中心になります。



---





### 使用可能なデータセット
決勝用として、9種類のデータセットを用意しました。
環境変数 `SFT_DATASET_ID` を変更することで切り替えられます。
  - この標準コードではデフォルトで1（ALFWorld）を使用しています。
  - さらなる性能向上のため、データセットに追加で前処理を行ってから学習を行っても差し支えありません。

1. ALFWorld：家庭内タスク  
[u-10bei/sft_alfworld_trajectory_dataset](https://huggingface.co/datasets/u-10bei/sft_alfworld_trajectory_dataset)  
[u-10bei/sft_alfworld_trajectory_dataset_v2](https://huggingface.co/datasets/u-10bei/sft_alfworld_trajectory_dataset_v2)  
[u-10bei/sft_alfworld_trajectory_dataset_v3](https://huggingface.co/datasets/u-10bei/sft_alfworld_trajectory_dataset_v3)  
[u-10bei/sft_alfworld_trajectory_dataset_v4](https://huggingface.co/datasets/u-10bei/sft_alfworld_trajectory_dataset_v4)   
 [u-10bei/sft_alfworld_trajectory_dataset_v5](https://huggingface.co/datasets/u-10bei/sft_alfworld_trajectory_dataset_v5)
2. DBBench：データベース操作  
[u-10bei/dbbench_sft_dataset_react](https://huggingface.co/datasets/u-10bei/dbbench_sft_dataset_react)  
[u-10bei/dbbench_sft_dataset_react_v2](https://huggingface.co/datasets/u-10bei/dbbench_sft_dataset_react_v2)  
[u-10bei/dbbench_sft_dataset_react_v3](https://huggingface.co/datasets/u-10bei/dbbench_sft_dataset_react_v3)  
[u-10bei/dbbench_sft_dataset_react_v4](https://huggingface.co/datasets/u-10bei/dbbench_sft_dataset_react_v4)  

In [6]:
import os
import random
import json
import shutil
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple

from datasets import load_dataset, Dataset
from transformers import TrainingArguments, Trainer, TrainerCallback


In [7]:
# -----------------------------
# 環境変数の設定
# -----------------------------
# 下記の値を書き換えることで、コード本体を編集せずに設定を変更できます。

# 1. モデル・データセット関連
os.environ["SFT_BASE_MODEL"] = "Qwen/Qwen3-4B-Instruct-2507"
#os.environ["SFT_BASE_MODEL"] = "Qwen/Qwen2.5-7B-Instruct"

os.environ["SFT_DATASET_ID"] = "u-10bei/dbbench_sft_dataset_react_v4"  # 自作データセットを使う場合はここを変更
os.environ["SFT_OUT_LORA_DIR"] = "/content/lora_agentbench_qwen3_4b"

# 2. 学習の基本パラメータ
os.environ["SFT_SEED"] = "3407"
os.environ["SFT_VAL_RATIO"] = "0.05"
os.environ["SFT_MAX_SEQ_LEN"] = "2048"#"1024"

# 3. LoRA (アダプタ) 設定
os.environ["SFT_LORA_R"] = "16"#32 #64
os.environ["SFT_LORA_ALPHA"] = "32"#64 #128
os.environ["SFT_LORA_DROPOUT"] = "0"
os.environ["SFT_LORA_TARGET_MODULES"] = "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"

# 4. ハイパーパラメータ
os.environ["SFT_EPOCHS"] = "2"
os.environ["SFT_PER_DEVICE_TRAIN_BS"] = "2"
os.environ["SFT_PER_DEVICE_EVAL_BS"] = "2"
os.environ["SFT_GRAD_ACCUM"] = "4"
os.environ["SFT_LR"] = "3e-6" #"5e-6"DBBenchはすでに高い完成度にあるため、学習率をさらに少し下げることで、ALFWorldの「元々の賢さ」を壊さないように慎重に微調整します。
#1e-4#2e-6#1e-5 フォーマットが崩れるのは「事前学習モデルが元々持っていた命令追従能力が、強すぎるSFTによって上書き破壊された（破局的忘却）」サインでもあります。現在の 1e-5 から 5e-6 に下げるのは、過学習を防ぎ、ベースモデルの賢さを残すための安全で定石的なアプローチです。
os.environ["SFT_WARMUP_RATIO"] = "0.03"
os.environ["SFT_WEIGHT_DECAY"] = "0.01"#0.05

# 5. ステップ・保存設定
os.environ["SFT_MAX_STEPS"] = "-1" # -1でエポックベース。動作確認時は 10 などに。
os.environ["SFT_LOGGING_STEPS"] = "10"
os.environ["SFT_EVAL_STEPS"] = "30"
os.environ["SFT_SAVE_STEPS"] = "100"
os.environ["SFT_SAVE_TOTAL_LIMIT"] = "2"

# 6. 特殊学習設定 (CoTマスク・アップサンプリング)
os.environ["SFT_MASK_COT"] = "1" # "1" で有効, "0" で無効 #0
os.environ["SFT_OUTPUT_MARKERS"] = "Output:"
#os.environ["SFT_OUTPUT_MARKERS"] = "Output:,OUTPUT:,Final:,Answer:,Result:,Response:" 。現状の設定（Final:,Answer:,Result:,Response:）だと、モデルは回答の前にこれらのマーカーを付ける癖を学習してしまいます。
os.environ["SFT_OUTPUT_LEARN_MODE"] = "after_marker" # "after_marker" または "from_marker"
os.environ["SFT_USE_UPSAMPLING"] = "0" # "1" で有効, "0" で無効  # データ2 専用 モデルが「推論環境でも思考プロセスをそのままアクションとして出力してしまう」現象が起きています。MASK_COT = "1" にすることで、思考部分の損失（Loss）を計算せず、最終的な「アクションの出力」にフォーカスして学習させることができるため、余計なおしゃべりが減ります。
os.environ["SFT_UPSAMPLE_RULES"] = "" # 例: '{"pack:math": 2.0}' # データ2 専用

print("環境変数の設定が完了しました。")

環境変数の設定が完了しました。


In [8]:
def _getenv(name: str, default: str):
    return os.environ.get(name, default)

def _getenv_int(name: str, default: int) -> int:
    try:
        return int(os.environ.get(name, str(default)))
    except Exception:
        return default

def _getenv_float(name: str, default: float) -> float:
    try:
        return float(os.environ.get(name, str(default)))
    except Exception:
        return default

# 学習の“出発点”となるベースモデル（4B）
BASE_MODEL_ID = _getenv("SFT_BASE_MODEL", "Qwen/Qwen3-4B-Instruct-2507")

# 学習に使うSFTデータセット（HF Hub上に置かれている想定）
DATASET_ID    = _getenv("SFT_DATASET_ID", "u-10bei/sft_alfworld_trajectory_dataset_v5")

# 学習後に保存されるLoRAアダプタの出力先（ローカル）
OUT_LORA_DIR  = _getenv("SFT_OUT_LORA_DIR", "/content/lora_agentbench_qwen3_4b") # HFアップロードするアダプタ名と合わせる

SEED        = _getenv_int("SFT_SEED", 3407)
VAL_RATIO   = _getenv_float("SFT_VAL_RATIO", 0.05)

# 1サンプルあたり最大何トークンまで見るか（長いほど情報を見られるが、GPUメモリと時間が増える）
MAX_SEQ_LEN = _getenv_int("SFT_MAX_SEQ_LEN", 2048)

# LoRA Config（＝“どれくらいの表現力を持つ差分を学習するか”）
LORA_R       = _getenv_int("SFT_LORA_R", 64)
LORA_ALPHA   = _getenv_int("SFT_LORA_ALPHA", 128)
LORA_DROPOUT = _getenv_float("SFT_LORA_DROPOUT", 0)
LORA_TARGET_MODULES = (
    _getenv("SFT_LORA_TARGET_MODULES", "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj").split(",")
)

# Train hyperparams（学習の基本設定）
NUM_TRAIN_EPOCHS            = _getenv_int("SFT_EPOCHS", 1)
PER_DEVICE_TRAIN_BATCH_SIZE = _getenv_int("SFT_PER_DEVICE_TRAIN_BS", 2)
PER_DEVICE_EVAL_BATCH_SIZE  = _getenv_int("SFT_PER_DEVICE_EVAL_BS", 2)

# 勾配累積：GPUに一度に載せられるバッチが小さい時に、複数ステップ分を貯めて“大きいバッチ相当”にする
GRAD_ACCUM                  = _getenv_int("SFT_GRAD_ACCUM", 4)

LR                          = _getenv_float("SFT_LR", 1e-6)
WARMUP_RATIO                = _getenv_float("SFT_WARMUP_RATIO", 0.1)

# Debug / quick check
# MAX_STEPSを小さくすると“動作確認だけ”の短時間学習ができます（本番は -1 のまま）
MAX_STEPS        = _getenv_int("SFT_MAX_STEPS", -1)
LOGGING_STEPS    = _getenv_int("SFT_LOGGING_STEPS", 10)
EVAL_STEPS       = _getenv_int("SFT_EVAL_STEPS", 30)
SAVE_STEPS       = _getenv_int("SFT_SAVE_STEPS", 100)
SAVE_TOTAL_LIMIT = _getenv_int("SFT_SAVE_TOTAL_LIMIT", 2)
WEIGHT_DECAY     = _getenv_float("SFT_WEIGHT_DECAY", 0.05)

# Optional: upsampling rules
# 特定のサブカテゴリ（例：難しいタスク）を“多めに学習させる”ための仕組み。
# 標準ではOFFになっています。
UPSAMPLE_ENABLE      = _getenv("SFT_USE_UPSAMPLING", "0") in ("1","true","True")
UPSAMPLE_RULES_JSON = _getenv("SFT_UPSAMPLE_RULES", "")


# -----------------------------
# 2.2) Seed & Utils
# -----------------------------
# 乱数（シャッフルやサンプリング）を固定して、再現性を担保します。
# seedが同じなら、原則として同じ分割・同じ抽出になりやすいです。

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# データセットをmessages形式に変換する関数
def convert_to_messages_format(example):
    """
    データセット形式に応じて適切にmessages形式に変換する。

    - 新データセット（trajectory形式）: messagesカラムがあればそのまま使用
    - 旧データセット（Q&A形式）: question/SQL から messages を構築
    """
    # 既にmessages形式ならそのまま返す（trajectory形式データセット）
    if "messages" in example and isinstance(example["messages"], list):
        msgs = example["messages"]
        # messages内の各要素が {role, content} を持つか確認
        if len(msgs) > 0 and isinstance(msgs[0], dict) and "role" in msgs[0]:
            if not hasattr(convert_to_messages_format, "_logged"):
                print("--- Debug: Dataset already in messages format (trajectory) ---")
                print(f"  Turns: {len(msgs)}")
                print(f"  Roles: {[m['role'] for m in msgs]}")
                convert_to_messages_format._logged = True
            return {"messages": msgs}

    # 旧形式: question/SQL or instruction/output から変換
    q = example.get('question') or example.get('instruction', "")
    inp = example.get('input', "")
    if inp and isinstance(inp, str) and inp.strip():
        q += f"\nSchema: {inp}"

    a = example.get('SQL') or example.get('output', "")

    if not hasattr(convert_to_messages_format, "_logged"):
        print("-- Debug: Converting from legacy format --")
        print(f"Q: {str(q)[:100]}...")
        print(f"A: {str(a)[:100]}...")
        convert_to_messages_format._logged = True

    user_content = f"You are a database expert. Your task is to generate a SQL query based on the provided question.\nQuestion: {q}"
    assistant_content = f"Output: {a}"

    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content}
        ]
    }

def ensure_openai_messages(ds: Dataset, msg_col: str = "messages") -> None:
    # データが「messages: [{role, content}, ...]」形式かをチェックします。
    # これは ChatGPT形式（OpenAIのChat Completions形式に似た）で、
    # tokenizer.apply_chat_template で安全に文字列化するために必要です。
    row0 = ds[0]
    ex = row0.get(msg_col, None)
    if not isinstance(ex, list):
        raise ValueError(f"Dataset must have list-style 'messages'. Got {type(ex)}")

def has_any_nonempty_assistant_turn(msgs: List[Dict[str, Any]]) -> bool:
    # “assistantの発話が空じゃない”ものが1回でも含まれるか？
    # SFTでは「正解例（assistantの出力）」がないと学習できないため。
    return any(m.get("role") == "assistant" and str(m.get("content", "")).strip() != "" for m in msgs)

def ends_with_nonempty_assistant(ex: Dict[str, Any]) -> bool:
    # トラジェクトリデータセットでは最後のメッセージがtoolであることがあるため、
    # 厳密に「最後のターンがassistant」である必要はないと判断し、
    # 学習可能なassistantターンが一つでもあればTrueを返すように変更します。
    # (has_any_nonempty_assistant_turnが既にこのチェックを行っています)
    # これにより、ValueError: num_samples=0 の問題を回避します。
    msgs = ex.get("messages", [])
    if not msgs:
        return False # No messages, not valid

    # 会話全体の中で空でないアシスタントターンがあるかどうかを確認します。
    # 少なくとも 1 つあり、照合者がすべてのアシスタントターンを処理する場合、
    # この例はトレーニングに使用できます。
    return has_any_nonempty_assistant_turn(msgs)

def shuffle_split(ds: Dataset, val_ratio: float, seed: int) -> Tuple[Dataset, Dataset]:
    # データをシャッフルして train/val に分割します。
    # val（検証）を持つことで「学習が進むほど性能が上がっているか／過学習していないか」を見られます。
    ds_shuf = ds.shuffle(seed=seed)
    n = len(ds_shuf)
    n_val = max(1, int(round(n * val_ratio)))
    return ds_shuf.select(range(n_val, n)), ds_shuf.select(range(n_val))

def make_text_cache_builder(tokenizer):
    # messages形式 → 実際にモデルに入力する“1本のテキスト”へ変換する関数を作ります。さらに「トークン長（truncationなし）」もキャッシュします。
    #
    # full_text  : ユーザー＋アシスタント（正解）まで含んだ全文
    # prefix_text: “最後のassistantの直前まで”の文（＝ここからassistantを生成させたい）
    #
    # この2つを持つことで、後のcollatorで「assistant部分だけをloss対象にする境界」を計算できます。

    def _build(batch):
        full_out = []
        prefix_out = []
        full_len_out = []
        prefix_len_out = []

        for msgs in batch["messages"]:
            full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False)
            prefix = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)

            full_out.append(full)
            prefix_out.append(prefix)

            # 重要：ここで truncation=False で token 長だけ計算してキャッシュする
            # add_special_tokens=False はあなたの現行設計に合わせる（テンプレ側で必要トークンが入る想定）
            full_ids = tokenizer(full, add_special_tokens=False, truncation=False)["input_ids"]
            prefix_ids = tokenizer(prefix, add_special_tokens=False, truncation=False)["input_ids"]

            full_len_out.append(len(full_ids))
            prefix_len_out.append(len(prefix_ids))

        return {
            "full_text": full_out,
            "prefix_text": prefix_out,
            "full_input_ids_len": full_len_out,
            "prefix_input_ids_len": prefix_len_out,
        }

    return _build



# -----------------------------
# 2.3) Collator (assistant-only loss)
# -----------------------------
# collatorは「生のサンプル群 → 学習に必要なテンソル(input_ids/labels等)」に変換する部品です。
#
# ここがこの学習コードの“設計思想”の核心：
# - 入力（user/system）も含めてモデルには読ませる
# - ただし loss（誤差）を計算するのは assistant の出力部分だけ
#
# これにより：
# - 「プロンプトを丸暗記させる」方向に学習が引っ張られにくい
# - “回答の形式”や“出力の正確さ”に学習の力点を置ける
#
# ALFWorldやDBBenchのようなマルチターンtool-useタスクでは、この設計は合理的です。

# 使用データセットによる仕様の違い
# データセット1：Output: が 100% なので CoT マスクが常に動き、Output本体だけ学習
# データセット2：Output: 系ラベルが存在しないため、CoTマスクは発動せず、“出力本体”を学習

# --- CoT mask settings (env overridable) ---
MASK_COT = _getenv("SFT_MASK_COT", "1") in ("1","true","True")
OUTPUT_MARKERS = [s.strip() for s in _getenv(
    "SFT_OUTPUT_MARKERS",
    "Output:,OUTPUT:,Final:,Answer:,Result:,Response:"
).split(",") if s.strip()]
OUTPUT_LEARN_MODE = _getenv("SFT_OUTPUT_LEARN_MODE", "after_marker")  # after_marker / from_marker

import torch
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class AssistantOnlyCollatorCached:
    """
    全てのassistant turnにlossをかけるCollator。

    旧版との違い:
    - 旧: 最後のassistant turnのみloss → trajectory中間ステップが学習されない
    - 新: 全assistant turnにloss → 環境観察、アクション選択、SQL構築、リカバリーを全て学習

    動作原理:
    1. 各サンプルのmessagesを chat_template で1本のテキストに変換
    2. 各assistant turnの開始・終了位置を特定
    3. assistant部分のみlabelsを設定、それ以外は-100（loss計算外）
    """
    tokenizer: Any
    max_length: int = 2048

    def _apply_template(self, msgs, tools=None, add_generation_prompt=False):
        """apply_chat_template のラッパー。tools と enable_thinking を統一的に渡す。"""
        kwargs = dict(
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,  # SFT では <think> ブロックを挿入しない
        )
        # tools がある場合のみ渡す（無い場合に渡すと template がエラーになる場合がある）
        if tools:
            kwargs["tools"] = tools
        return self.tokenizer.apply_chat_template(msgs, **kwargs)

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        tok = self.tokenizer

        all_input_ids = []
        all_labels = []
        all_attention = []

        for ex in batch:
            msgs = ex["messages"]
            tools = ex.get("tools", None)  # function-calling 形式のツール定義

            # 全メッセージを1本のテキストに変換（tools 付き）
            full_text = self._apply_template(
                msgs, tools=tools, add_generation_prompt=False
            )

            # truncation なしの全長を取得（境界計算用）
            full_ids_notrun = tok(
                full_text, add_special_tokens=False,
                truncation=False
            )["input_ids"]

            # truncation ありでエンコード（実際の入力用）
            full_ids = tok(
                full_text, add_special_tokens=False,
                truncation=True, max_length=self.max_length
            )["input_ids"]

            # left-truncationのオフセット
            trunc_offset = max(0, len(full_ids_notrun) - self.max_length)

            labels = [-100] * len(full_ids)

            # 各 assistant turn の境界を計算してlabelsを設定
            for i, msg in enumerate(msgs):
                if msg["role"] != "assistant":
                    continue

                # msgs[0..i-1] のテキスト長 = このassistant turnの開始位置
                prefix_text = self._apply_template(
                    msgs[:i], tools=tools, add_generation_prompt=True
                )
                prefix_ids = tok(
                    prefix_text, add_special_tokens=False, truncation=False
                )["input_ids"]

                # msgs[0..i] のテキスト長 = このassistant turnの終了位置
                through_text = self._apply_template(
                    msgs[:i+1], tools=tools, add_generation_prompt=False
                )
                through_ids = tok(
                    through_text, add_special_tokens=False, truncation=False
                )["input_ids"]

                # truncation補正後の位置
                start = max(0, len(prefix_ids) - trunc_offset)
                end = max(0, len(through_ids) - trunc_offset)

                # labelsにassistant部分のtoken IDをコピー
                for j in range(start, min(end, len(full_ids))):
                    if j >= 0:
                        labels[j] = full_ids[j]

            all_input_ids.append(full_ids)
            all_labels.append(labels)
            all_attention.append([1] * len(full_ids))

        # Padding (right padding)
        max_len = max(len(ids) for ids in all_input_ids)
        pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0

        padded_ids = []
        padded_labels = []
        padded_attention = []

        for i in range(len(all_input_ids)):
            pad_len = max_len - len(all_input_ids[i])
            padded_ids.append(all_input_ids[i] + [pad_id] * pad_len)
            padded_labels.append(all_labels[i] + [-100] * pad_len)
            padded_attention.append(all_attention[i] + [0] * pad_len)

        return {
            "input_ids": torch.tensor(padded_ids, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long),
            "attention_mask": torch.tensor(padded_attention, dtype=torch.long),
        }

In [9]:
import random, torch
from datasets import Dataset
from huggingface_hub import hf_hub_download
import json

@torch.no_grad()
def filter_has_supervision(ds, collator):
    keep = []
    for i in range(len(ds)):
        out = collator([ds[i]])
        if (out["labels"][0] != -100).sum().item() > 0:
            keep.append(i)
    return ds.select(keep)


def count_all_masked(ds, collator, n=200, seed=3407):
    rng = random.Random(seed)
    n = min(n, len(ds))
    idxs = [rng.randrange(0, len(ds)) for _ in range(n)]
    all_masked = 0
    for i in idxs:
        out = collator([ds[i]])
        labels = out["labels"][0]
        if (labels != -100).sum().item() == 0:
            all_masked += 1
    print(f"[CHECK] all-masked samples in {n}: {all_masked} ({all_masked/max(1,n):.1%})")

# -----------------------------
# 2.4) Callback (monitor)
# -----------------------------
# 学習中のデバッグ用コールバックです。
# ここでは「labelsのうち、実際にloss対象になっているトークン割合」を時々表示します。
#
# 意味：
# - valid_ratio が極端に小さい → “学習していない”のと同じ（ラベルがほぼ -100）
# - valid_ratio が適度にある → assistant部分にしっかりlossが乗っている
#
# 初学者向けに言うと：
# - これは“学習がちゃんと効いているかの健康診断”です。

class LabelStatsCallback(TrainerCallback):
    def __init__(self, dataset, collator, name="train", every_n_steps=100):
        self.dataset, self.collator, self.name, self.every_n_steps = dataset, collator, name, every_n_steps

    @torch.no_grad()
    def on_step_end(self, args, state, control, **kwargs):
        if (state.global_step % self.every_n_steps) == 0:
            batch = [self.dataset[random.randint(0, len(self.dataset)-1)] for _ in range(8)]
            out = self.collator(batch)
            valid = (out["labels"] != -100).sum().item()
            total = (out["attention_mask"] == 1).sum().item()
            print(f"\n[LabelStats:{self.name}] step={state.global_step} valid_ratio={valid/max(1,total):.4f}")


# -----------------------------
# 2.5) Main
# -----------------------------
# main() が実行されると、ここまでの部品を使って学習が一気に進みます。

def main():
    os.makedirs(OUT_LORA_DIR, exist_ok=True)

    # if you used /content/your_id cache dirs etc, remove to avoid confusion
    if os.path.exists("/content/your_id"):
        shutil.rmtree("/content/your_id")

    print(f"[INFO] Loading dataset from HF Hub: {DATASET_ID}")

    try:
        # First, try standard loading
        ds_all = load_dataset(DATASET_ID, split="train", token=HF_TOKEN)
    except Exception as e:
        print(f"[WARN] Standard load failed: {e}\nTrying manual loading (download -> parse -> normalize keys)...")
        try:
            # 手動ダウンロード
            # Note: ファイル名はデータセットによって異なります。必要に応じて変更してください。
            # ファイル名はデータセットに応じて変更してください
            file_path = hf_hub_download(repo_id=DATASET_ID, filename="data.jsonl", repo_type="dataset", token=HF_TOKEN)

            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            data_list = []
            for line in lines:
                if line.strip():
                    data_list.append(json.loads(line))

            if not data_list:
                raise ValueError("Downloaded file is empty.")

            # キーを正規化: すべての辞書に見つかったキーの和集合が含まれるようにします
            all_keys = set().union(*(d.keys() for d in data_list))
            for d in data_list:
                for k in all_keys:
                    if k not in d:
                        d[k] = None

            ds_all = Dataset.from_list(data_list)
            print(f"[INFO] Manual load successful. {len(ds_all)} rows loaded.")

        except Exception as e2:
            print(f"[ERROR] Manual load also failed: {e2}")
            raise e  # フォールバックが失敗した場合は元のエラーを発生させる

    print(f"[INFO] Loaded dataset columns: {ds_all.column_names}")

    # データセットをmessages形式に変換する (既存のnotebookから移動)
    print("Converting dataset to 'messages' format...")

    # converted_dsの初期化
    converted_ds = ds_all

    # messagesカラムが既にあるか確認
    if "messages" in ds_all.column_names:
       # trajectory形式: messages と tools を保持し、他のカラムを削除
       keep_cols = {"messages", "tools"}  # tools はfunction-calling形式で必要
       remove_cols = [c for c in ds_all.column_names if c not in keep_cols]
       if remove_cols:
           converted_ds = ds_all.remove_columns(remove_cols)
       has_tools = "tools" in ds_all.column_names
       print(f"[INFO] Dataset already has 'messages' column. tools={'present' if has_tools else 'absent'}.")
    else:
       # 旧形式: 変換が必要
       print("[INFO] Converting dataset to messages format...")
       converted_ds = ds_all.map(convert_to_messages_format, remove_columns=ds_all.column_names)

    print("Conversion complete. Displaying first converted sample:")
    if len(converted_ds) > 0:
        sample_messages = converted_ds[0]['messages']
        print(json.dumps(sample_messages, indent=2, ensure_ascii=False))
    else:
        print("Converted dataset is empty.")

    ds_all = converted_ds # Update ds_all for subsequent steps
    print(f"New dataset columns: {ds_all.column_names}")

    # データ形式チェック（messagesがlistであること）
    ensure_openai_messages(ds_all)

    # 基本的なフィルタリング（アシスタントが存在する必要があり、最後のターンはアシスタント）
    # 学習できるサンプルだけ残す（assistantが空なら教師信号が無い）
    ds_all = ds_all.filter(lambda ex: has_any_nonempty_assistant_turn(ex["messages"]))
    ds_all = ds_all.filter(ends_with_nonempty_assistant)

    # train/val分割
    train_ds, val_ds = shuffle_split(ds_all, VAL_RATIO, SEED)

    print("[INFO] Loading base model:", BASE_MODEL_ID)

    # Unslothでベースモデルを読み込む（4bitロードで省メモリ）
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=False,
    )

    # Cache chat template renders
    # ※現行Collatorはmessagesから直接境界計算するため、このキャッシュは
    #   Collator内部では参照されません。将来の拡張やデバッグ用に残しています。
    build_cache = make_text_cache_builder(tokenizer)
    train_ds = train_ds.map(build_cache, batched=True, num_proc=1, desc="Caching train")
    val_ds   = val_ds.map(build_cache,   batched=True, num_proc=1, desc="Caching val")

    # Attach LoRA
    # ここで「学習される部分（LoRAアダプタ）」をモデルに追加します。
    # 学習対象は LoRA のパラメータだけになり、ベースモデルの巨大な重みは固定されます。
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=LORA_TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    # IMPORTANT FIX:
    # transformers TrainingArguments uses eval_strategy (NOT evaluation_strategy)
    # ※ここは重要：Transformersの引数名がバージョンで揺れることがあります。
    # 今回のバージョンでは eval_strategy を使います。
    args = TrainingArguments(
        output_dir=OUT_LORA_DIR,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,

        logging_steps=LOGGING_STEPS,

        eval_strategy="steps",
        eval_steps=EVAL_STEPS,

        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,

        max_steps=MAX_STEPS,  # -1 => epoch-based

        bf16=True,             # L4はbf16ネイティブ対応
        fp16=False,

        push_to_hub=False,
        report_to="none",

        group_by_length=False,
        remove_unused_columns=False,
    )

    # assistant-only loss の collator を使う
    collator = AssistantOnlyCollatorCached(tokenizer=tokenizer, max_length=MAX_SEQ_LEN)

    # --- NaN対策：all-masked（教師トークン0）を除去して評価を安定化 ---
    print("[INFO] Checking all-masked samples before filtering...")
    count_all_masked(val_ds, collator, n=len(val_ds), seed=SEED)

    print("[INFO] Filtering train/val to remove all-masked samples...")
    train_ds = filter_has_supervision(train_ds, collator)
    val_ds   = filter_has_supervision(val_ds, collator)

    print("[INFO] New sizes:", "train =", len(train_ds), "val =", len(val_ds))
    print("[INFO] Checking all-masked samples after filtering...")
    count_all_masked(val_ds, collator, n=len(val_ds), seed=SEED)


    # Trainer（Transformersの標準学習ループ）
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        tokenizer=tokenizer,
    )

    # 監視用コールバックを追加（学習が効いているかのヘルスチェック）
    trainer.add_callback(LabelStatsCallback(train_ds, collator, name="train", every_n_steps=LOGGING_STEPS))

    print("[INFO] Starting training...")
    trainer.train()

    # 学習後の保存：LoRAアダプタ＆tokenizer
    # ただし各自の再利用や追加学習、共有のために保存しておくことをお勧めします。
    print("[INFO] Saving adapter & tokenizer...")
    model.save_pretrained(OUT_LORA_DIR)
    tokenizer.save_pretrained(OUT_LORA_DIR)
    print(f"[INFO] Done. Saved to {OUT_LORA_DIR}")

if __name__ == "__main__":
    main()

[INFO] Loading dataset from HF Hub: u-10bei/dbbench_sft_dataset_react_v4


README.md: 0.00B [00:00, ?B/s]

dbbench_sft_data_react_v4.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

[INFO] Loaded dataset columns: ['id', 'messages', 'metadata']
Converting dataset to 'messages' format...
[INFO] Dataset already has 'messages' column. tools=absent.
Conversion complete. Displaying first converted sample:
[
  {
    "role": "user",
    "content": "\nI will ask you a question, then you should help me operate a MySQL database with SQL to answer the question.\nYou have to explain the problem and your solution to me and write down your thoughts.\nAfter thinking and explaining thoroughly, every round you can choose to operate or to answer.\nyour operation should be like this:\nAction: Operation\n```sql\nSELECT * FROM table WHERE condition;\n```\nYou MUST put SQL in markdown format without any other comments. Your SQL should be in one line.\nEvery time you can only execute one SQL statement. I will only execute the statement in the first SQL code block. Every time you write a SQL, I will execute it for you and give you the output.\nIf you are done operating, and you want to co

...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
30,1.714000,1.504074
60,0.892000,0.859290
90,0.686900,0.652172
120,0.515200,0.536438
150,0.465400,0.465613
180,0.416400,0.417491
210,0.368700,0.390935
240,0.403800,0.379557
270,0.380600,0.375313



[LabelStats:train] step=10 valid_ratio=0.2634

[LabelStats:train] step=20 valid_ratio=0.2786


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



[LabelStats:train] step=30 valid_ratio=0.2623

[LabelStats:train] step=40 valid_ratio=0.2609

[LabelStats:train] step=50 valid_ratio=0.2815

[LabelStats:train] step=60 valid_ratio=0.2691

[LabelStats:train] step=70 valid_ratio=0.2756

[LabelStats:train] step=80 valid_ratio=0.2399

[LabelStats:train] step=90 valid_ratio=0.2821

[LabelStats:train] step=100 valid_ratio=0.2577

[LabelStats:train] step=110 valid_ratio=0.2700

[LabelStats:train] step=120 valid_ratio=0.2779

[LabelStats:train] step=130 valid_ratio=0.2703

[LabelStats:train] step=140 valid_ratio=0.2887

[LabelStats:train] step=150 valid_ratio=0.2654

[LabelStats:train] step=160 valid_ratio=0.2799

[LabelStats:train] step=170 valid_ratio=0.2766

[LabelStats:train] step=180 valid_ratio=0.2341

[LabelStats:train] step=190 valid_ratio=0.2765

[LabelStats:train] step=200 valid_ratio=0.2670

[LabelStats:train] step=210 valid_ratio=0.2893

[LabelStats:train] step=220 valid_ratio=0.2504

[LabelStats:train] step=230 valid_ratio=0.2518

## Step 4: 学習成果物の確認と、LoRAアダプタのhuggingfaceへのアップロード
学習後、OUT_LORA_DIR に以下が保存されます（最低限）ので、確認してください。
- adapter_config.json
- adapter_model.safetensors（または adapter_model.bin）
- tokenizer 関連ファイル
<br>

下記に従って"README.md"を記載してから、HuggingFaceにアダプタアップロードを実行してください。



### ① README.md の正しい書き方

#### README.md の役割（最重要）

Hugging Face では **README.md = モデルカード**です。
「このLoRAは何を学習し、どう使い、何に注意すべきか」を **第三者が再利用できる水準で説明する義務**があります。

README が不十分なモデルは、

* OSSとして不適切
* 学習内容が不透明
* ライセンス違反リスクあり
  と評価されます。

---

#### 必須構成（この順で書くこと）

##### 1. YAMLメタデータ（必須）

```yaml
---
base_model: Qwen/Qwen3-4B-Instruct-2507
datasets:
- u-10bei/sft_alfworld_trajectory_dataset_v5
- u-10bei/sft_sql_dataset_trajectories_v5
language:
- en
license: MIT
library_name: peft
pipeline_tag: text-generation
tags:
- lora
- agent
- tool-use
- alfworld
- dbbench
---
```

**理由**

* HF検索・分類・再現性に必須
* 無いと「壊れたモデルカード」扱いになる

---

##### 2. モデル概要（What）

```md
# qwen3-4b-agent-trajectory-lora

This repository provides a **LoRA adapter** fine-tuned from
**Qwen3-4B-Instruct-2507** using **LoRA + Unsloth**.

This repository contains **LoRA adapter weights only**.
The base model must be loaded separately.
```

**必須ポイント**

* 「LoRAアダプタのみ」であることを明記
* ベースモデル名を明示

---

##### 3. 学習目的・設計思想（Why）

```md
## Training Objective

This adapter is trained to improve **multi-turn agent task performance**
on ALFWorld (household tasks) and DBBench (database operations).

Loss is applied to **all assistant turns** in the multi-turn trajectory,
enabling the model to learn environment observation, action selection,
tool use, and recovery from errors.
```

**今回の講座では特に重要**

* 全assistant turn に loss をかける設計
* マルチターン tool-use パターンの学習

---

##### 4. 学習設定（How）

```md
## Training Configuration

- Base model: Qwen3-4B-Instruct-2507
- Method: LoRA (full precision base)
- Max sequence length: 2048
- Epochs: 1
- Learning rate: 2e-6
- LoRA: r=64, alpha=128
```

**再現性のため必須**

---

##### 5. 使用方法（How to use）

````md
## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = "Qwen/Qwen3-4B-Instruct-2507"
adapter = "your-name/your-repo"

tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(
    base,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter)
````

````

---

##### 6. データセット・ライセンス注意（必須・重要）
```md
## Sources & Terms (IMPORTANT)

Training data:
- u-10bei/sft_alfworld_trajectory_dataset_v5
- u-10bei/sft_sql_dataset_trajectories_v5

This repository does NOT redistribute the dataset.
Users must comply with the dataset license and base model terms.
````


## Load and Merge Models

### Subtask:
Unslothを使用して、訓練済みLoRAアダプタをベースモデルにロードし、マージします。これにより、完全な推論可能なモデルが生成されます。


In [10]:
import os
from google.colab import userdata
from unsloth import FastLanguageModel
from huggingface_hub import login
import torch # torch.bfloat16 を使用するためにインポート

# HF_TOKEN の取得とログイン
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
  #  HF_TOKEN = userdata.get('HFH_FG') # Colab秘密鍵から取得
     HF_TOKEN = userdata.get('HF_TOKEN') # Colab秘密鍵から取得

if not HF_TOKEN:
    print("警告: HF_TOKENが見つかりません。Hugging Faceへのログインが失敗する可能性があります。")
else:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("Logged into Hugging Face.")

print(f"\n[INFO] Loading base model: {BASE_MODEL_ID}")

# 3. ベースモデルのロード
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16, # full精度（bfloat16）を明示的に指定
    load_in_4bit=False,   # 4bit量子化を無効化
)

print("[INFO] Applying LoRA configuration...")
# 4. モデルにLoRA設定を適用
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print(f"[INFO] Loading LoRA adapter weights from: {OUT_LORA_DIR}")
# 5. 学習済みLoRAアダプタの重みをロード
model.load_adapter(OUT_LORA_DIR, adapter_name="default")

print("[INFO] Merging LoRA adapter with the base model...")
# 6. LoRAアダプタをベースモデルにマージ
model = model.merge_and_unload()

print("[INFO] Model successfully loaded and merged.")
print("You can now use 'model' for inference.")

Logged into Hugging Face.

[INFO] Loading base model: Qwen/Qwen3-4B-Instruct-2507
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[INFO] Applying LoRA configuration...
[INFO] Loading LoRA adapter weights from: /content/lora_agentbench_qwen3_4b
[INFO] Merging LoRA adapter with the base model...
[INFO] Model successfully loaded and merged.
You can now use 'model' for inference.


### 補足
前のステップでLoRAアダプタのロードとベースモデルへのマージが完了しました。
次のステップで、マージ済みモデルをローカルに保存します。

In [11]:
import os

# 前のステップで定義された変数を引き継ぐ設定
# ノートブック内で先に定義されていない場合に備え、デフォルト値を設定

def _getenv(name: str, default: str):
    return os.environ.get(name, default)

MERGED_MODEL_DIR = _getenv("MERGED_MODEL_DIR", "/content/merged_model")

# 4. マージ済みモデルとトークナイザーをローカルに保存
print(f"Saving merged model and tokenizer to {MERGED_MODEL_DIR}...")
os.makedirs(MERGED_MODEL_DIR, exist_ok=True)
model.save_pretrained(MERGED_MODEL_DIR)
tokenizer.save_pretrained(MERGED_MODEL_DIR)
print("Merged model and tokenizer saved locally.")

Saving merged model and tokenizer to /content/merged_model...
Merged model and tokenizer saved locally.


In [12]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [13]:
import os
import shutil

# コピー元ディレクトリ
source_dir = '/content/merged_model'

# コピー先ディレクトリ (Google Drive内のパスを適切に設定してください)
# 例: Google Drive の 'My Drive' 直下に保存する場合
target_dir = '/content/drive/MyDrive/Colab Notebooks/LLD_FINAL_ADVANCE_ORI/Model/rev.05'

# ターゲットディレクトリが存在しない場合は作成
os.makedirs(target_dir, exist_ok=True)

# ディレクトリの内容をコピー
# shutil.copytree はディレクトリ自体とその内容をコピーします。
# 既にディレクトリが存在する場合はエラーになるため、事前に削除するか、
# os.makedirs(target_dir, exist_ok=True) を使い、内容のみコピーする場合は
# 個別のファイルコピー処理を実装する必要があります。
# 今回は、target_dir が空であることを前提に copytree を使用します。

# 既存のターゲットディレクトリを削除してからコピーする場合:
if os.path.exists(target_dir):
    print(f"既存のディレクトリ {target_dir} を削除します...")
    shutil.rmtree(target_dir)

print(f"'{source_dir}' から '{target_dir}' へファイルをコピーします...")
shutil.copytree(source_dir, target_dir)
print("コピーが完了しました。")

既存のディレクトリ /content/drive/MyDrive/Colab Notebooks/LLD_FINAL_ADVANCE_ORI/Model/rev.05 を削除します...
'/content/merged_model' から '/content/drive/MyDrive/Colab Notebooks/LLD_FINAL_ADVANCE_ORI/Model/rev.05' へファイルをコピーします...
コピーが完了しました。


### コピー元ディレクトリの内容を確認
`/content/merged_model` の内容を表示します。

In [ ]:
import os

source_dir = '/content/merged_model'
print(f"--- Contents of {source_dir} ---")
for root, dirs, files in os.walk(source_dir):
    level = root.replace(source_dir, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')


--- Contents of /content/merged_model ---


### コピー先ディレクトリの内容を確認
Google Drive上の `/content/drive/MyDrive/Colab Notebooks/LLD_FINAL_ADVANCE_ORI/Model/rev.0` の内容を表示します。

In [ ]:
import os

target_dir = '/content/drive/MyDrive/Colab Notebooks/LLD_FINAL_ADVANCE_ORI/Model/rev.0'
print(f"--- Contents of {target_dir} ---")
if not os.path.exists(target_dir):
    print(f"Warning: Target directory {target_dir} does not exist. It might not have been created or copied correctly.")
else:
    for root, dirs, files in os.walk(target_dir):
        level = root.replace(target_dir, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')


--- Contents of /content/drive/MyDrive/Colab Notebooks/LLD_FINAL_ADVANCE_ORI/Model/rev.0 ---


## Upload Merged Model to Hugging Face

### Subtask:
保存されたマージ済みモデルをHugging Faceリポジトリ`u-10bei/test01`にアップロードします。README.mdはアップロード時に自動で生成されるデフォルトの内容となります。


In [14]:
# -----------------------------
# README.md（モデルカード）を OUT_LORA_DIR に生成
# -----------------------------
# 学習完了後に実行し、Hugging Face の README.md（モデルカード）を生成
# ベースモデル名・データセット名・学習ハイパーパラメータはコードの変数から自動同期

import os

os.makedirs(OUT_LORA_DIR, exist_ok=True)

# ------------------------------------------------------------------
# 補助関数の定義
# ------------------------------------------------------------------
def _s(x, default=""):
    try:
        v = str(x)
        return v if v.strip() else default
    except Exception:
        return default

def _fmt_lr(x) -> str:
    """
    Learning Rate の表記を整えるための関数。

    - 数値として解釈できる場合：
      指数表記（例: 1e-6）に整形する
    - 数値として解釈できない場合：
      元の値をそのまま文字列として出力する
      （誤った値を生成しないための安全策）
    """
    try:
        return f"{float(x):.0e}"
    except Exception:
        return _s(x, "")


# ------------------------------------------------------------------
# 学習コードの変数から値を取得（README と自動同期）
# ------------------------------------------------------------------
base_model_id = _s(BASE_MODEL_ID, "Qwen/Qwen3-4B-Instruct-2507")
dataset_id = _s(DATASET_ID, "u-10bei/sft_alfworld_trajectory_dataset_v5")

max_seq_len = int(MAX_SEQ_LEN)
epochs = int(NUM_TRAIN_EPOCHS)
lr_str = _fmt_lr(LR)

lora_r = int(LORA_R)
lora_alpha = int(LORA_ALPHA)

# NOTE:
# - YAML front matter の license は
#   「この LoRA アダプタ（リポジトリ）のライセンス表明」を意味する。
# - 必要に応じて環境変数で差し替え可能。
repo_license = os.environ.get("SFT_REPO_LICENSE", "apache-2.0")

# README 内に記載するモデルタイトル
# 変更したい場合は README.md を手書きで調整
title_line = "qwen3-4b-agent-trajectory-lora-SFT-SQL-ALFWorld_rev.0.5" #例： qwen3-4b-agent-trajectory-lora

# ------------------------------------------------------------------
# README.md 本文の生成
# （説明テキストに準拠し、変数部分のみを自動置換）
# ------------------------------------------------------------------
readme_md = f"""---
base_model: {base_model_id}
datasets:
- {dataset_id}
language:
- en
license: {repo_license}
library_name: peft
pipeline_tag: text-generation
tags:
- lora
- agent
- tool-use
- alfworld
- dbbench
---

# {title_line}

This repository provides a **LoRA adapter** fine-tuned from
**{base_model_id}** using **LoRA + Unsloth**.

This repository contains **LoRA adapter weights only**.
The base model must be loaded separately.

## Training Objective

This adapter is trained to improve **multi-turn agent task performance**
on ALFWorld (household tasks) and DBBench (database operations).

Loss is applied to **all assistant turns** in the multi-turn trajectory,
enabling the model to learn environment observation, action selection,
tool use, and recovery from errors.

## Training Configuration

- Base model: {base_model_id}
- Method: LoRA (full precision base)
- Max sequence length: {max_seq_len}
- Epochs: {epochs}
- Learning rate: {lr_str}
- LoRA: r={lora_r}, alpha={lora_alpha}

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = "{base_model_id}"
adapter = "your_id/your-repo"

tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(
    base,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter)
```

## Sources & Terms (IMPORTANT)

Training data: {dataset_id}

Dataset License: MIT License. This dataset is used and distributed under the terms of the MIT License.
Compliance: Users must comply with the MIT license (including copyright notice) and the base model's original terms of use.
"""
# ------------------------------------------------------------------
# README.md の書き込み
# ------------------------------------------------------------------

readme_path = os.path.join(OUT_LORA_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_md)

# ------------------------------------------------------------------
# 動作確認
# ------------------------------------------------------------------

assert os.path.exists(readme_path), "README.md was not written."
assert readme_md.lstrip().startswith("---\n"), (
    "README.md must start with YAML front matter."
)
# 修正: 先頭の --- は改行なしで始まるため count("\n---\n") には含まれない。
# そのため、閉じタグの分として 1回以上あればOKとする。
assert readme_md.count("\n---\n") >= 1, (
    "YAML front matter must be closed properly."
)

print(f"[INFO] README.md written to: {readme_path}")
print("[INFO] Preview (first 30 lines):")
for i, line in enumerate(readme_md.splitlines()[:30], start=1):
    print(f"{i:02d}: {line}")

[INFO] README.md written to: /content/lora_agentbench_qwen3_4b/README.md
[INFO] Preview (first 30 lines):
01: ---
02: base_model: Qwen/Qwen3-4B-Instruct-2507
03: datasets:
04: - u-10bei/dbbench_sft_dataset_react_v4
05: language:
06: - en
07: license: apache-2.0
08: library_name: peft
09: pipeline_tag: text-generation
10: tags:
11: - lora
12: - agent
13: - tool-use
14: - alfworld
15: - dbbench
16: ---
17: 
18: # qwen3-4b-agent-trajectory-lora-SFT-SQL-ALFWorld_rev.0.5
19: 
20: This repository provides a **LoRA adapter** fine-tuned from
21: **Qwen/Qwen3-4B-Instruct-2507** using **LoRA + Unsloth**.
22: 
23: This repository contains **LoRA adapter weights only**.
24: The base model must be loaded separately.
25: 
26: ## Training Objective
27: 
28: This adapter is trained to improve **multi-turn agent task performance**
29: on ALFWorld (household tasks) and DBBench (database operations).
30: 


### 補足
マージ済みモデルのローカル保存が完了したら、以下のコードでHugging Faceにアップロードします。
アップロード先のリポジトリIDは適宜変更してください。

In [15]:
import os
from huggingface_hub import HfApi, upload_folder
import shutil # shutil をインポートしてファイル操作を可能にします

# HF_UPLOAD_REPO_ID, MERGED_MODEL_DIR, PRIVATE_REPO, HF_TOKEN が
# 前のステップで定義済みであることを前提とします。

# アップロード先の設定
HF_UPLOAD_REPO_ID = "Kumeichi/qwen3-4b-agent-trajectory-lora-SFT-SQL-ALFWorld_rev.0.5" # マージ済みモデルのアップロード先
PRIVATE_REPO = False # リポジトリを非公開にする場合は True に変更

api = HfApi()

# リポジトリが存在しない場合は作成
print(f"Creating/checking Hugging Face repository: {HF_UPLOAD_REPO_ID}...")
api.create_repo(
    repo_id=HF_UPLOAD_REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=PRIVATE_REPO,
)
print("Hugging Face repository is ready.")

# OUT_LORA_DIR に生成された README.md を MERGED_MODEL_DIR にコピー
readme_src_path = os.path.join(OUT_LORA_DIR, "README.md")
readme_dst_path = os.path.join(MERGED_MODEL_DIR, "README.md")

if os.path.exists(readme_src_path):
    shutil.copy(readme_src_path, readme_dst_path)
    print(f"Copied README.md from {readme_src_path} to {readme_dst_path}")
else:
    print(f"Warning: README.md not found at {readme_src_path}. Uploading without it.")

# マージ済みモデルのフォルダをアップロード
print(f"Uploading merged model to Hugging Face repository: {HF_UPLOAD_REPO_ID}...")
upload_folder(
    folder_path=MERGED_MODEL_DIR,
    repo_id=HF_UPLOAD_REPO_ID,
    repo_type="model",
    commit_message="Upload merged Qwen3-4B-Instruct-2507 model (auto-generated README)",
    token=HF_TOKEN,
)
print("Merged model upload complete.")
print(f"You can view your model at: https://huggingface.co/{HF_UPLOAD_REPO_ID}")

Creating/checking Hugging Face repository: Kumeichi/qwen3-4b-agent-trajectory-lora-SFT-SQL-ALFWorld_rev.0.5...
Hugging Face repository is ready.
Copied README.md from /content/lora_agentbench_qwen3_4b/README.md to /content/merged_model/README.md
Uploading merged model to Hugging Face repository: Kumeichi/qwen3-4b-agent-trajectory-lora-SFT-SQL-ALFWorld_rev.0.5...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  603kB / 3.08GB            

  ...0001-of-00002.safetensors:   1%|          | 33.5MB / 4.97GB            

  ...rged_model/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Merged model upload complete.
You can view your model at: https://huggingface.co/Kumeichi/qwen3-4b-agent-trajectory-lora-SFT-SQL-ALFWorld_rev.0.5
